**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# SNR=50 Scatter Plot Evaluation

Generates **predicted vs. ground-truth scatter plots** at **SNR=50** for three methods:

| Row | Method | Colour |
|-----|--------|---------|
| 1 | Dictionary Matching (DM) | Orange (`#E65100`) |
| 2 | DL — Noise-Free trained (Triple A+B+C NF) | Green (`#2E7D32`) |
| 3 | DL — Noisy trained (Triple A+B+C Mixed-SNR) | Blue (`#1565C0`) |

**Requires** (relative paths, adjust `PATHS` in cell 2 if needed):
- `../subsamples/subsamples_v3/QuasiRand_t2_200.mat`   — noise-free dictionary
- `../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat` — parameters
- `../echotimes.mat`
- `./triple_regime_nf_results_v1/models/triple_nf_best.pt`   — NF model checkpoint
- `./triple_regime_results_v1/models/triple_regime_best.pt`  — Noisy model checkpoint


## 1. Imports & device

In [ ]:
import os, json, time
import numpy as np
import scipy.io as sio
import h5py
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import torch
import torch.nn as nn

# ── Plot style (matches reference notebooks) ──────────────────────────────
plt.rcParams.update({
    'font.family'    : 'Arial',
    'font.size'      : 9,
    'axes.labelsize' : 10,
    'axes.titlesize' : 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'     : 150,
    'savefig.dpi'    : 300,
    'savefig.bbox'   : 'tight',
})

# ── Colour palette (matches reference notebooks exactly) ──────────────────
C_DM       = '#E65100'   # deep orange  — DM
C_DL_NF    = '#2E7D32'   # deep green   — DL noise-free
C_DL_NOISY = '#1565C0'   # deep blue    — DL noisy / triple A+B+C

# ── Device ────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    device = torch.device('cpu')
    print('CPU only')
print(f'PyTorch {torch.__version__}')

## 2. Configuration — **edit paths here**

In [ ]:
CONFIG = {
    # ── Data paths ────────────────────────────────────────────────────────
    'noisefree_sig_path' : '../subsamples/subsamples_v3/QuasiRand_t2_200.mat',
    'param_path'         : '../subsamples/subsamples_v3/QuasiRand_par_t2_200.mat',
    'echotimes_path'     : '../echotimes.mat',
    'dict_key'           : 'Dico40_save',
    'param_key'          : 'par_save',

    # ── Model checkpoints ────────────────────────────────────────────────
    'ckpt_nf'    : './results/triple_regime_nf_results_v1/models/triple_nf_best.pt',
    'ckpt_noisy' : './results/triple_regime_results_v1/models/triple_regime_best.pt',

    # ── Output ────────────────────────────────────────────────────────────
    'output_dir' : './results/snr50_scatter_eval',

    # ── Parameter space ───────────────────────────────────────────────────
    'param_mins'  : np.array([0.0,    0.0025,  1.0e-6,  0.050]),
    'param_maxs'  : np.array([1.0,    0.15,   25.0e-6,  0.200]),
    'param_names' : ['SO2', 'CBV', 'R', 'T2'],

    # ── GESFIDE geometry ─────────────────────────────────────────────────
    'n_fid'   : 14,
    'n_rephas': 16,
    'n_postse': 10,

    # ── Triple-regime feature scaling ─────────────────────────────────────
    'R2starA_min':  2.0,   'R2starA_max': 55.0,
    'R2starB_min': -30.0,  'R2starB_max': 22.0,
    'R2starC_min':  2.0,   'R2starC_max': 55.0,

    # ── Test set ─────────────────────────────────────────────────────────
    'test_snr'   : 50,
    'n_test'     : 100_000,    # number of test samples (subsample if dict is larger)
    'n_scatter'  : 15_000,     # points shown in scatter (for speed/readability)

    # ── DM ────────────────────────────────────────────────────────────────
    # DM dictionary: same noise-free dict (normalised), matched by inner product
    'dm_subsample': 200_000,   # subsample dict for DM to limit RAM; set None for full
}

SE_ECHO = CONFIG['n_fid'] + CONFIG['n_rephas']  # = 30
os.makedirs(CONFIG['output_dir'], exist_ok=True)
FIG_DIR = os.path.join(CONFIG['output_dir'], 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

PARAM_NAMES = ['SO\u2082',  'CBV',  'R',    'T2']
PARAM_UNITS = ['(%)',       '(%)',  '(\u00b5m)', '(ms)']
PARAM_SCALE = [100,          100,    1e6,     1000]
PARAM_LABELS = [f'{n} {u}' for n, u in zip(PARAM_NAMES, PARAM_UNITS)]

print(f'SE_ECHO = {SE_ECHO}')
print(f'Test SNR: {CONFIG["test_snr"]}')
print(f'Output: {CONFIG["output_dir"]}')

## 3. Utilities

In [ ]:
def load_mat(path, key):
    """Load a .mat variable, handles both v7.3 (HDF5) and older formats."""
    try:
        mat = sio.loadmat(path)
        if key in mat:
            return np.array(mat[key], dtype=np.float32)
        cands = [k for k in mat if not k.startswith('_')]
        print(f'  Key "{key}" not found, using "{cands[0]}"')
        return np.array(mat[cands[0]], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            data = f[key][()] if key in f else f[next(k for k in f if not k.startswith('#'))][()]
            if data.ndim >= 2: data = data.T
            return np.array(data, dtype=np.float32)


def euclidean_norm(data):
    data = np.abs(data).astype(np.float32)
    return data / np.maximum(np.linalg.norm(data, axis=1, keepdims=True), 1e-12)


def params_scale(p, mins, maxs):
    return ((p - mins) / (maxs - mins)).astype(np.float32)


def params_inverse(p, mins, maxs):
    return (p * (maxs - mins) + mins).astype(np.float32)


def filter_param_range(signals, params, mins, maxs):
    mask = np.ones(len(params), dtype=bool)
    for i in range(min(params.shape[1], len(mins))):
        mask &= (params[:, i] >= mins[i]) & (params[:, i] <= maxs[i])
    if (~mask).sum():
        print(f'  Filtered {(~mask).sum():,} out-of-range ({(~mask).mean()*100:.1f}%)')
    return signals[mask], params[mask]


def clean_data(signals, params):
    valid = np.all(np.isfinite(signals), axis=1) & np.all(np.isfinite(params), axis=1)
    if (~valid).sum(): print(f'  Removed {(~valid).sum():,} non-finite entries')
    return signals[valid], params[valid]


def add_rician_noise(sig_nf, snr, rng=None):
    """Add Rician noise at the given SNR level.  sig_nf shape: (N, echoes)."""
    if rng is None: rng = np.random.default_rng(0)
    S0    = np.abs(sig_nf[:, 0:1])          # reference amplitude from first echo
    sigma = S0 / snr
    nr    = rng.normal(0, 1, sig_nf.shape).astype(np.float32) * sigma
    ni    = rng.normal(0, 1, sig_nf.shape).astype(np.float32) * sigma
    return np.sqrt((sig_nf + nr)**2 + ni**2).astype(np.float32)


def batched_predict(model, x_np, batch_size=4096, dev=device):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(x_np), batch_size):
            xb = torch.tensor(x_np[i:i+batch_size], dtype=torch.float32).to(dev).contiguous()
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, axis=0)


def save_fig(fig, name):
    for ext in ['pdf', 'png']:
        p = os.path.join(FIG_DIR, f'{name}.{ext}')
        fig.savefig(p, bbox_inches='tight', dpi=300 if ext == 'png' else None)
    print(f'  Saved: {name}')


print('Utilities ready.')

## 4. Load echo times

In [ ]:
et_mat       = sio.loadmat(CONFIG['echotimes_path'])
echo_times_s = et_mat['Echotimes'].flatten() / 1000.0

T_A     = echo_times_s[:CONFIG['n_fid']]
T_B     = echo_times_s[CONFIG['n_fid']:SE_ECHO]
T_C     = echo_times_s[SE_ECHO:]
T_SE_S  = echo_times_s[SE_ECHO - 1]
T_C_rel = T_C - T_SE_S

print(f'Part A: echoes  0-{CONFIG["n_fid"]-1},   t = [{T_A[0]*1e3:.2f},...,{T_A[-1]*1e3:.2f}] ms')
print(f'Part B: echoes {CONFIG["n_fid"]}-{SE_ECHO-1},   t = [{T_B[0]*1e3:.2f},...,{T_B[-1]*1e3:.2f}] ms')
print(f'Part C: echoes {SE_ECHO}-{SE_ECHO+CONFIG["n_postse"]-1},  t_rel = [{T_C_rel[0]*1e3:.2f},...,{T_C_rel[-1]*1e3:.2f}] ms')
print(f'Spin echo at {T_SE_S*1e3:.2f} ms')

## 5. Triple-regime feature functions

In [ ]:
def ols_slope(t_vec, sig_mat):
    """OLS slope of log|S| vs t. Returns raw slope (N,)."""
    log_s = np.log(np.maximum(np.abs(sig_mat), 1e-9)).astype(np.float64)
    t     = t_vec.astype(np.float64)
    t_c   = t - t.mean()
    log_sm = log_s - log_s.mean(axis=1, keepdims=True)
    return (log_sm * t_c[None, :]).sum(axis=1) / (t_c ** 2).sum()


def compute_triple_regime_features(sig_raw, config):
    """
    R2*_A = R2 + R2'  (Part A, always positive)
    R2*_B = R2 - R2'  (Part B, raw signed)
    R2*_C = R2 + R2'  (Part C, relative to SE, always positive)
    """
    n_fid   = config['n_fid']
    se_echo = n_fid + config['n_rephas']
    r2_lb   = 1.0 / config['param_maxs'][3]

    slope_A = ols_slope(T_A,     sig_raw[:, :n_fid])
    slope_B = ols_slope(T_B,     sig_raw[:, n_fid:se_echo])
    slope_C = ols_slope(T_C_rel, sig_raw[:, se_echo:])

    R2starA = np.maximum(-slope_A, r2_lb).astype(np.float32)
    R2starB = (-slope_B).astype(np.float32)   # raw signed — no clip
    R2starC = np.maximum(-slope_C, r2_lb).astype(np.float32)
    return R2starA, R2starB, R2starC


def scale_triple_features(R2starA, R2starB, R2starC, config):
    """Min-max scale all three regime features to [0, 1]."""
    def sc(x, lo, hi):
        return np.clip((x - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)
    return (sc(R2starA, config['R2starA_min'], config['R2starA_max']),
            sc(R2starB, config['R2starB_min'], config['R2starB_max']),
            sc(R2starC, config['R2starC_min'], config['R2starC_max']))


def build_43dim_input(sig_raw, config):
    """Raw signal → 43-dim model input: [L2-norm signal (40)] + [feat_A, feat_B, feat_C]."""
    R2A, R2B, R2C = compute_triple_regime_features(sig_raw, config)
    fA, fB, fC    = scale_triple_features(R2A, R2B, R2C, config)
    sig_norm       = euclidean_norm(sig_raw)
    return np.concatenate([sig_norm, fA[:, None], fB[:, None], fC[:, None]], axis=1)


print('Triple-regime feature functions ready.')

## 6. Model architecture (TripleRegimeModel)

In [ ]:
class Clamp01(nn.Module):
    def forward(self, x): return x.clamp(0.0, 1.0)


class FiLMLayer(nn.Module):
    def __init__(self, feature_dim, cond_in=3, cond_hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_in, cond_hidden), nn.ReLU(),
            nn.Linear(cond_hidden, 2 * feature_dim),
        )
        nn.init.zeros_(self.net[-1].weight)
        b = torch.zeros(2 * feature_dim)
        b[:feature_dim] = 1.0
        self.net[-1].bias.data.copy_(b)
        self.feature_dim = feature_dim

    def forward(self, x, cond):
        p = self.net(cond)
        return p[:, :self.feature_dim] * x + p[:, self.feature_dim:]


class TripleRegimeModel(nn.Module):
    """
    Conv1D backbone + FiLM conditioning on (R2*_A, R2*_B, R2*_C).
    Input:  (B, 43)  Output: (B, 4)
    """
    def __init__(self, n_outputs=4, dropout=0.05, film_cond_hidden=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32,  7, padding=3), nn.BatchNorm1d(32),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64),  nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(64, 128,3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128,256,3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256,256,3, padding=1), nn.BatchNorm1d(256), nn.ReLU(),
        )
        ch = film_cond_hidden
        self.fc1   = nn.Linear(1280, 512); self.bn1 = nn.BatchNorm1d(512)
        self.film1 = FiLMLayer(512, 3, ch)
        self.fc2   = nn.Linear(512,  256); self.bn2 = nn.BatchNorm1d(256)
        self.film2 = FiLMLayer(256, 3, ch)
        self.fc3   = nn.Linear(256,  128); self.bn3 = nn.BatchNorm1d(128)
        self.film3 = FiLMLayer(128, 3, ch)
        self.fc_out  = nn.Linear(128, n_outputs)
        self.out_act = Clamp01()
        self.drop    = nn.Dropout(dropout)
        self.relu    = nn.ReLU()

    def forward(self, x):
        echo = x[:, :40].unsqueeze(1)
        cond = x[:, 40:43]
        c = self.conv(echo).flatten(1)
        h = self.drop(self.relu(self.bn1(self.film1(self.fc1(c), cond))))
        h = self.drop(self.relu(self.bn2(self.film2(self.fc2(h), cond))))
        h = self.drop(self.relu(self.bn3(self.film3(self.fc3(h), cond))))
        return self.out_act(self.fc_out(h))


# Quick shape check
_m = TripleRegimeModel().to(device)
_x = torch.randn(8, 43).to(device)
assert _m(_x).shape == (8, 4), 'Shape mismatch!'
print(f'TripleRegimeModel OK — {sum(p.numel() for p in _m.parameters()):,} parameters')
del _m, _x

## 7. Build SNR=50 test set

In [ ]:
print('Loading noise-free dictionary...')
sig_nf  = load_mat(CONFIG['noisefree_sig_path'], CONFIG['dict_key'])
par_all = load_mat(CONFIG['param_path'],         CONFIG['param_key'])[:, :4]

# Filter to valid physiological range
sig_nf, par_all = filter_param_range(sig_nf, par_all, CONFIG['param_mins'], CONFIG['param_maxs'])
sig_nf, par_all = clean_data(sig_nf, par_all)
print(f'Clean dictionary: {len(sig_nf):,} samples  shape={sig_nf.shape}')

# ── Subsample for test set ────────────────────────────────────────────────
n_test = min(CONFIG['n_test'], len(sig_nf))
rng = np.random.default_rng(42)
idx_test = rng.choice(len(sig_nf), n_test, replace=False)
sig_test_nf = sig_nf[idx_test]          # (n_test, 40)  — noise-free signals
par_test    = par_all[idx_test]          # (n_test, 4)   — ground truth params

# ── Add Rician noise at SNR=50 ────────────────────────────────────────────
SNR_TEST = CONFIG['test_snr']
sig_test_noisy = add_rician_noise(sig_test_nf, SNR_TEST, rng=np.random.default_rng(99))

# ── Scale ground truth to physical units for plotting ────────────────────
# par_test columns: [SO2 (0-1), CBV (0-0.15), R (1e-6 to 25e-6 m), T2 (0.05-0.2 s)]
par_test_phys = par_test.copy()
for i, sc in enumerate(PARAM_SCALE):
    par_test_phys[:, i] = par_test[:, i] * sc

print(f'\nTest set: {n_test:,} samples at SNR={SNR_TEST}')
print(f'  SO2:  [{par_test_phys[:,0].min():.1f}, {par_test_phys[:,0].max():.1f}] %')
print(f'  CBV:  [{par_test_phys[:,1].min():.3f}, {par_test_phys[:,1].max():.1f}] %')
print(f'  R:    [{par_test_phys[:,2].min():.2f}, {par_test_phys[:,2].max():.1f}] µm')
print(f'  T2:   [{par_test_phys[:,3].min():.0f}, {par_test_phys[:,3].max():.0f}] ms')

## 8. Dictionary Matching (DM) — inner product

In [ ]:
print('Running Dictionary Matching (DM) on GPU...')
t0 = time.time()

# Subsample dictionary
dm_sub = CONFIG.get('dm_subsample')
if dm_sub is not None and dm_sub < len(sig_nf):
    rng_dm = np.random.default_rng(7)
    idx_dm = rng_dm.choice(len(sig_nf), dm_sub, replace=False)
    dm_dict_sig = sig_nf[idx_dm]
    dm_dict_par = par_all[idx_dm]
    print(f'  DM dictionary subsampled: {dm_sub:,} entries')
else:
    dm_dict_sig = sig_nf
    dm_dict_par = par_all

# Move normalised arrays to GPU
dm_dict_norm_gpu = torch.tensor(euclidean_norm(dm_dict_sig), dtype=torch.float32).to(device)  # (D, 40)
test_norm_gpu    = torch.tensor(euclidean_norm(sig_test_noisy), dtype=torch.float32).to(device)  # (N, 40)

# Chunked inner-product on GPU
CHUNK = 4096
dm_pred_idx = np.empty(n_test, dtype=np.int32)
with torch.no_grad():
    for start in range(0, n_test, CHUNK):
        end = min(start + CHUNK, n_test)
        ip  = test_norm_gpu[start:end] @ dm_dict_norm_gpu.T  # (chunk, D)
        dm_pred_idx[start:end] = ip.argmax(dim=1).cpu().numpy()

dm_pred_par = dm_dict_par[dm_pred_idx]

# Convert to physical units
dm_pred_phys = dm_pred_par.copy()
for i, sc in enumerate(PARAM_SCALE):
    dm_pred_phys[:, i] = dm_pred_par[:, i] * sc

print(f'  DM done in {time.time()-t0:.1f}s')

## 9. DL — Noise-Free model (Triple A+B+C NF)

In [ ]:
print('Loading DL Noise-Free model...')
model_nf = TripleRegimeModel(n_outputs=4, dropout=0.05).to(device)
state_nf = torch.load(CONFIG['ckpt_nf'], map_location=device)
model_nf.load_state_dict(state_nf)
model_nf.eval()
print(f'  Loaded: {CONFIG["ckpt_nf"]}')

# Build 43-dim input from NOISY test signals
print('  Building 43-dim input (noisy test signals → features)...')
x_test_nf_model = build_43dim_input(sig_test_noisy, CONFIG)

# Clean any NaN from feature computation
valid_nf = np.all(np.isfinite(x_test_nf_model), axis=1)
print(f'  Valid samples: {valid_nf.sum():,} / {n_test:,}')

# Predict
t0 = time.time()
nf_pred_scaled = batched_predict(model_nf, x_test_nf_model)
nf_pred_raw    = params_inverse(nf_pred_scaled, CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4])

# Convert to physical units
nf_pred_phys = nf_pred_raw.copy()
for i, sc in enumerate(PARAM_SCALE):
    nf_pred_phys[:, i] = nf_pred_raw[:, i] * sc

print(f'  DL-NF inference done in {time.time()-t0:.1f}s')

## 10. DL — Noisy model (Triple A+B+C Mixed-SNR)

In [ ]:
print('Loading DL Noisy (Mixed-SNR) model...')
model_noisy = TripleRegimeModel(n_outputs=4, dropout=0.05).to(device)
state_noisy = torch.load(CONFIG['ckpt_noisy'], map_location=device)
model_noisy.load_state_dict(state_noisy)
model_noisy.eval()
print(f'  Loaded: {CONFIG["ckpt_noisy"]}')

# Re-use the same 43-dim input computed from noisy signals
# (x_test_nf_model was already built from sig_test_noisy)
t0 = time.time()
noisy_pred_scaled = batched_predict(model_noisy, x_test_nf_model)
noisy_pred_raw    = params_inverse(noisy_pred_scaled, CONFIG['param_mins'][:4], CONFIG['param_maxs'][:4])

# Convert to physical units
noisy_pred_phys = noisy_pred_raw.copy()
for i, sc in enumerate(PARAM_SCALE):
    noisy_pred_phys[:, i] = noisy_pred_raw[:, i] * sc

print(f'  DL-Noisy inference done in {time.time()-t0:.1f}s')

## 11. Compute metrics

In [ ]:
def compute_metrics(true_phys, pred_phys, method_name):
    """Compute R², RMSE, bias for all 4 parameters."""
    metrics = {}
    print(f"\n{'='*65}")
    print(f'{method_name}')
    print(f"{'Parameter':<8} {'RMSE':>10} {'Bias':>10} {'R²':>8}")
    print(f"{'-'*65}")
    for i, (name, unit) in enumerate(zip(PARAM_NAMES, PARAM_UNITS)):
        t = true_phys[:, i]
        p = pred_phys[:, i]
        v = np.isfinite(t) & np.isfinite(p)
        if v.sum() < 10:
            metrics[name] = {'rmse': np.nan, 'bias': np.nan, 'r2': np.nan}
            continue
        rmse = float(np.sqrt(np.mean((t[v]-p[v])**2)))
        bias = float(np.mean(p[v] - t[v]))
        r2   = float(r2_score(t[v], p[v]))
        metrics[name] = {'rmse': rmse, 'bias': bias, 'r2': r2}
        print(f"{name:<8} {rmse:>8.3f}{unit:>4}  {bias:>+8.3f}{unit:>4}  {r2:>8.4f}")
    print(f"{'='*65}")
    return metrics


metrics_dm    = compute_metrics(par_test_phys, dm_pred_phys,    f'Dictionary Matching (DM) — SNR={SNR_TEST}')
metrics_nf    = compute_metrics(par_test_phys, nf_pred_phys,    f'DL Noise-Free (Triple A+B+C NF) — SNR={SNR_TEST}')
metrics_noisy = compute_metrics(par_test_phys, noisy_pred_phys, f'DL Noisy/Mixed-SNR (Triple A+B+C) — SNR={SNR_TEST}')

## 12. Scatter plot — 3 rows × 4 columns

In [ ]:
def make_scatter_row(axes, true_phys, pred_phys, colour, row_label, n_scatter=15_000):
    """
    Fill one row of scatter subplots (4 parameters).
    Style matches the reference figure: blue dots, orange fit line, black dashed identity.
    """
    rng_sc = np.random.default_rng(0)
    idx_sc = rng_sc.choice(len(true_phys), min(n_scatter, len(true_phys)), replace=False)

    for ci, (ax, pname, punit) in enumerate(zip(axes, PARAM_NAMES, PARAM_UNITS)):
        t = true_phys[idx_sc, ci]
        p = pred_phys[idx_sc, ci]
        v = np.isfinite(t) & np.isfinite(p)

        # Scatter
        ax.scatter(t[v], p[v], s=1.5, alpha=0.10, color=colour, rasterized=True)

        # Axis range
        all_t = true_phys[:, ci]
        lo = np.nanpercentile(all_t, 0.5)
        hi = np.nanpercentile(all_t, 99.5)
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)

        # Identity line
        ax.plot([lo, hi], [lo, hi], 'k--', lw=1.0, alpha=0.6)

        # Linear fit
        if v.sum() > 10:
            m, b  = np.polyfit(t[v], p[v], 1)
            xf    = np.array([lo, hi])
            ax.plot(xf, m*xf + b, color='#E65100', lw=1.5, alpha=0.9)

            # Metrics on full test set (not just scatter subset)
            tv_all = true_phys[:, ci]
            pv_all = pred_phys[:, ci]
            vv     = np.isfinite(tv_all) & np.isfinite(pv_all)
            r2_val = float(r2_score(tv_all[vv], pv_all[vv]))
            rmse_v = float(np.sqrt(np.mean((tv_all[vv] - pv_all[vv])**2)))

            ax.text(0.05, 0.93, f'R² = {r2_val:.3f}',
                    transform=ax.transAxes, fontsize=12, fontweight='bold',
                    color='#111111')
            # ax.text(0.05, 0.82, f'RMSE = {rmse_v:.2f}',
            #         transform=ax.transAxes, fontsize=7.5, color='#444444')

        ax.set_xlabel('True', fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(labelsize=8)

    # Y-axis label on first column only
    axes[0].set_ylabel(f'{row_label}\nPred', fontsize=8.5)


# ── Build figure ──────────────────────────────────────────────────────────
N_SCATTER = CONFIG['n_scatter']

fig, axes = plt.subplots(
    3, 4,
    figsize=(12, 8.5),
    gridspec_kw={'wspace': 0.38, 'hspace': 0.52}
)

# ── Row 0: DM ─────────────────────────────────────────────────────────────
make_scatter_row(axes[0], par_test_phys, dm_pred_phys,
                 colour=C_DL_NOISY, row_label='DM', n_scatter=N_SCATTER)

# ── Row 1: DL Noise-Free ──────────────────────────────────────────────────
make_scatter_row(axes[1], par_test_phys, nf_pred_phys,
                 colour=C_DL_NOISY, row_label='DL\nNoise-Free', n_scatter=N_SCATTER)

# ── Row 2: DL Noisy ───────────────────────────────────────────────────────
make_scatter_row(axes[2], par_test_phys, noisy_pred_phys,
                 colour=C_DL_NOISY, row_label='DL\nNoisy', n_scatter=N_SCATTER)

# ── Column titles (parameter names) ──────────────────────────────────────
for ci, label in enumerate(PARAM_LABELS):
    axes[0, ci].set_title(label, fontsize=11, fontweight='bold')

# ── Figure title ─────────────────────────────────────────────────────────
fig.suptitle(
    f'Predicted vs. Ground Truth — SNR = {SNR_TEST}\n'
    f'DM  |  DL Noise-Free (Triple A+B+C NF)  |  DL Noisy (Triple A+B+C)',
    fontsize=10, y=1.01
)

save_fig(fig, f'scatter_snr{SNR_TEST}_DM_DLnf_DLnoisy')
plt.show()
print('Done.')

## 13. Per-method individual figures (optional — same style, 1×4 each)

In [ ]:
methods = [
    ('DM',         dm_pred_phys,    C_DM,       'Dictionary Matching (DM)'),
    ('DL_NF',      nf_pred_phys,    C_DL_NF,    'DL Noise-Free — Triple (A+B+C)'),
    ('DL_Noisy',   noisy_pred_phys, C_DL_NOISY, 'DL Noisy — Triple (A+B+C)'),
]

for tag, pred_phys, colour, title in methods:
    fig1, axes1 = plt.subplots(1, 4, figsize=(11, 2.8),
                               gridspec_kw={'wspace': 0.38})
    make_scatter_row(axes1, par_test_phys, pred_phys,
                     colour=colour, row_label='Pred', n_scatter=CONFIG['n_scatter'])

    for ci, label in enumerate(PARAM_LABELS):
        axes1[ci].set_title(label, fontsize=11, fontweight='bold')

    fig1.suptitle(f'{title} — SNR = {SNR_TEST}', fontsize=10, y=1.04)
    save_fig(fig1, f'scatter_snr{SNR_TEST}_{tag}')
    plt.show()

print('All individual figures saved.')

## 14. Summary table

In [ ]:
print(f'\n{"="*75}')
print(f'SUMMARY — SNR={SNR_TEST} test set ({n_test:,} samples)')
print(f'{"="*75}')
print(f'{"":<22} {"  SO2 (%)": >12} {"  CBV (%)": >12} {"  R (µm)": >12} {"  T2 (ms)": >12}')
print(f'{"─"*75}')

for mname, mdict in [(f'DM (R²)',         metrics_dm),
                     (f'DL-NF (R²)',      metrics_nf),
                     (f'DL-Noisy (R²)',   metrics_noisy)]:
    vals = [f'{mdict[n]["r2"]:>11.3f}' for n in ["SO2", "CBV", "R", "T2"]]
    print(f'{mname:<22} {" ".join(vals)}')

print(f'{"─"*75}')
for mname, mdict in [(f'DM (RMSE)',       metrics_dm),
                     (f'DL-NF (RMSE)',    metrics_nf),
                     (f'DL-Noisy (RMSE)', metrics_noisy)]:
    vals = [f'{mdict[n]["rmse"]:>11.3f}' for n in ["SO2", "CBV", "R", "T2"]]
    print(f'{mname:<22} {" ".join(vals)}')

print(f'{"="*75}')

# Save as JSON
summary = {
    'snr': SNR_TEST,
    'n_test': n_test,
    'DM':       metrics_dm,
    'DL_NF':    metrics_nf,
    'DL_Noisy': metrics_noisy,
}
with open(os.path.join(CONFIG['output_dir'], f'metrics_snr{SNR_TEST}.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Metrics saved to {CONFIG["output_dir"]}/metrics_snr{SNR_TEST}.json')